# A07 — Choose-Your-Own-Adventure Story Engine

**NLP Lab · Creative & Generative · Ambitious**

This notebook implements an LLM-driven CYOA engine with **novel features** beyond a basic prompt loop:

| Feature | Purpose |
|---------|---------|
| **Narrative Anchors** | Memorable plot elements in `world_state.anchors`, scored for late callbacks |
| **Dual-layer memory** | Structured JSON + rolling `narrative_summary` each turn |
| **Continuity Guardian** | Rule checks after each turn (inventory, anchors, travel) |
| **Genre extensions** | Extra state for `fantasy`, `noir`, or `sci_fi` |
| **Playthrough export** | Markdown log for report / demo |

**Deliverables:** engine code · 10+ turn log · early→late callback · failure analysis


## 0. Setup

```bash
pip install -r requirements.txt
```

Set `OPENAI_API_KEY`, or `STORY_ENGINE_PROVIDER=ollama` with Ollama running.  
Without either, **mock mode** runs a deterministic 12-turn demo (continuity-friendly).


In [ ]:
%pip install -q openai python-dotenv


In [ ]:
import json
import os
from pathlib import Path
from IPython.display import display, Markdown

engine_path = Path("story_engine.py")
if not engine_path.exists():
    engine_path = Path("nlp-lab/A07_story_engine/story_engine.py")

exec(engine_path.read_text(encoding="utf-8"), globals())

provider = "openai" if os.getenv("OPENAI_API_KEY") else os.getenv("STORY_ENGINE_PROVIDER", "mock")
print("Backend:", provider)
print("Ready.")


## 1. Engine implementation

Core class: `StoryEngine` — see next cell (full source).


In [ ]:
"""
Choose-Your-Own-Adventure Story Engine (A07)
Novelty: Narrative Anchors + Dual-layer memory + Continuity Guardian
"""
from __future__ import annotations

import json
import os
import re
import textwrap
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Callable, Literal

try:
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    pass

Provider = Literal["openai", "ollama", "mock"]


def empty_world_state(genre: str = "fantasy") -> dict[str, Any]:
    return {
        "turn": 0,
        "genre": genre,
        "location": "unknown",
        "locations_visited": [],
        "characters_present": [],
        "inventory": [],
        "mood": "curious",
        "player_health": 100,
        "flags": {},
        "anchors": [],
        "last_choice": None,
        "extensions": _genre_extensions(genre),
    }


def _genre_extensions(genre: str) -> dict[str, Any]:
    if genre == "noir":
        return {"suspicion_level": 0, "clues_found": []}
    if genre == "sci_fi":
        return {"ship_integrity": 100, "credits": 50}
    return {"magic_affinity": 0, "quests_active": []}


@dataclass
class TurnRecord:
    turn: int
    scene: str
    choices: list[str]
    choice_taken: str | None
    world_state: dict[str, Any]
    narrative_summary: str
    continuity_notes: list[str] = field(default_factory=list)
    anchor_callbacks: list[str] = field(default_factory=list)


@dataclass
class PlaythroughLog:
    title: str
    genre: str
    started_at: str
    records: list[TurnRecord] = field(default_factory=list)

    def to_markdown(self) -> str:
        lines = [
            f"# Playthrough: {self.title}",
            f"- Genre: `{self.genre}`",
            f"- Started: {self.started_at}",
            f"- Turns: {len(self.records)}",
            "",
        ]
        for r in self.records:
            lines.append(f"## Turn {r.turn}")
            lines.append(r.scene.strip())
            lines.append("")
            lines.append("**Choices offered:**")
            for i, c in enumerate(r.choices, 1):
                lines.append(f"{i}. {c}")
            if r.choice_taken:
                lines.append(f"\n**Player chose:** {r.choice_taken}")
            lines.append("\n**World state:**")
            lines.append("```json")
            lines.append(json.dumps(r.world_state, indent=2))
            lines.append("```")
            if r.anchor_callbacks:
                lines.append(
                    f"\n*Anchor callbacks this turn:* {', '.join(r.anchor_callbacks)}"
                )
            if r.continuity_notes:
                lines.append(f"\n*Continuity notes:* {'; '.join(r.continuity_notes)}")
            lines.append("")
        return "\n".join(lines)


class StoryEngine:
    """
    LLM story engine with:
    - Structured world_state JSON updated every turn
    - Rolling narrative_summary (compressed memory)
    - Narrative anchors planted early, scored for callbacks later
    - Continuity guardian flags contradictions vs prior state
    """

    SYSTEM_PROMPT = textwrap.dedent(
        """\
        You are the game master for a choose-your-own-adventure text game.
        Each response MUST be valid JSON only (no markdown fences), with this schema:
        {
          "scene": "2-4 paragraphs of vivid second-person narration ending on a hook",
          "choices": ["choice A", "choice B", "choice C"],
          "world_state": { ...full updated state object... },
          "narrative_summary": "3-5 sentences summarizing the story so far for memory",
          "new_anchors": [{"id": "slug", "description": "memorable plot element", "introduced_turn": N}],
          "continuity_notes": ["optional notes about callbacks or consistency"]
        }

        Rules:
        - Exactly 3 distinct, meaningful choices.
        - Update world_state every turn: location, characters_present, inventory, mood,
          locations_visited, flags, extensions (genre-specific), and anchors list.
        - Merge new_anchors into world_state.anchors (do not drop old anchors).
        - From turn 3 onward, plant at least one vivid "anchor" (object, promise, or NPC trait).
        - From turn 8 onward, explicitly callback at least one anchor from turn <= 4 in the scene.
        - Keep inventory and location changes logically consistent with the player's last choice.
        - Player choice will be provided; honor it in the next scene.
        """
    )

    def __init__(
        self,
        title: str = "The Anchor Below",
        genre: str = "fantasy",
        provider: Provider | None = None,
        model: str | None = None,
    ):
        self.title = title
        self.genre = genre
        self.provider: Provider = provider or self._detect_provider()
        self.model = model or self._default_model()
        self.world_state = empty_world_state(genre)
        self.narrative_summary = "The adventure has not begun."
        self.log = PlaythroughLog(
            title=title,
            genre=genre,
            started_at=datetime.now(timezone.utc).isoformat(),
        )
        self._client: Any = None

    def _detect_provider(self) -> Provider:
        if os.getenv("STORY_ENGINE_PROVIDER", "").lower() == "mock":
            return "mock"
        if os.getenv("OPENAI_API_KEY"):
            return "openai"
        if os.getenv("STORY_ENGINE_PROVIDER", "").lower() == "ollama":
            return "ollama"
        return "mock"

    def _default_model(self) -> str:
        if self.provider == "openai":
            return os.getenv("STORY_ENGINE_MODEL", "gpt-4o-mini")
        if self.provider == "ollama":
            return os.getenv("STORY_ENGINE_MODEL", "llama3.2")
        return "mock-narrator-v1"

    def _get_client(self) -> Any:
        if self._client is not None:
            return self._client
        if self.provider == "openai":
            from openai import OpenAI

            self._client = OpenAI()
        elif self.provider == "ollama":
            from openai import OpenAI

            self._client = OpenAI(
                base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
                api_key="ollama",
            )
        return self._client

    def _call_llm(self, user_prompt: str) -> dict[str, Any]:
        if self.provider == "mock":
            return self._mock_response(user_prompt)
        client = self._get_client()
        resp = client.chat.completions.create(
            model=self.model,
            temperature=0.85,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": self.SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        )
        raw = resp.choices[0].message.content or "{}"
        return json.loads(raw)

    def _mock_response(self, user_prompt: str) -> dict[str, Any]:
        """Deterministic demo path for grading without API keys."""
        turn = self.world_state.get("turn", 0) + 1
        choice = self._extract_choice_from_prompt(user_prompt)
        ws = json.loads(json.dumps(self.world_state))

        if turn == 1:
            scene = (
                "You stand at the mouth of the Sunken Library, rain hissing on brass lanterns. "
                "A moth-eaten map is pressed into your palm—its ink smells of copper."
            )
            ws.update(
                location="Sunken Library entrance",
                locations_visited=["Sunken Library entrance"],
                inventory=["brass lantern", "copper-scented map"],
                mood="wary",
            )
            new_anchors = [
                {
                    "id": "copper_map",
                    "description": "Map whose ink smells of copper; marks a sealed stack",
                    "introduced_turn": 1,
                }
            ]
        elif turn <= 3:
            scene = (
                f"You act on: «{choice}». Deeper halls hum with drowned whispers. "
                "The copper map warms when you face west."
            )
            ws["locations_visited"].append("west corridor")
            ws["location"] = "west corridor"
            new_anchors = []
            if turn == 3:
                new_anchors.append(
                    {
                        "id": "librarian_promise",
                        "description": "Ghost librarian promised safe passage if you return the map",
                        "introduced_turn": 3,
                    }
                )
                ws["flags"]["librarian_deal"] = True
        elif turn >= 8:
            scene = (
                "At the sealed stack, the copper-scented map flares—the same scent from turn 1. "
                "The ghost librarian materializes: 'You kept our bargain.' Your inventory still "
                "holds the brass lantern that lit turn 2's path."
            )
            new_anchors = []
        else:
            scene = (
                f"Mid-adventure beat after «{choice}». Allies and hazards tighten around your mood: "
                f"{ws.get('mood')}."
            )
            new_anchors = []

        ws["turn"] = turn
        anchors = list(ws.get("anchors", []))
        for a in new_anchors:
            if not any(x.get("id") == a["id"] for x in anchors):
                anchors.append(a)
        ws["anchors"] = anchors
        ws["last_choice"] = choice

        return {
            "scene": scene,
            "choices": [
                "Follow the warming map west",
                "Question the ghost librarian",
                "Search your pack for another clue",
            ],
            "world_state": ws,
            "narrative_summary": f"Turn {turn}: {scene[:120]}...",
            "new_anchors": new_anchors,
            "continuity_notes": ["mock continuity path"],
        }

    @staticmethod
    def _extract_choice_from_prompt(prompt: str) -> str:
        m = re.search(r"Player chose:\s*(.+?)(?:\n|$)", prompt, re.I)
        return m.group(1).strip() if m else "explore cautiously"

    def _build_user_prompt(self, player_choice: str | None) -> str:
        self.world_state["turn"] = self.world_state.get("turn", 0) + 1
        turn = self.world_state["turn"]
        anchor_hint = ""
        if turn >= 8 and self.world_state.get("anchors"):
            early = [
                a
                for a in self.world_state["anchors"]
                if a.get("introduced_turn", 99) <= 4
            ]
            if early:
                anchor_hint = (
                    "\nREQUIRED: Reference at least one early anchor in the scene: "
                    + json.dumps(early)
                )
        choice_block = (
            f"Player chose: {player_choice}\n"
            if player_choice
            else "This is the opening scene. No prior player choice.\n"
        )
        return textwrap.dedent(
            f"""\
            Game: {self.title}
            Genre: {self.genre}
            Turn number: {turn}
            {choice_block}
            Current world_state:
            {json.dumps(self.world_state, indent=2)}

            Narrative summary so far:
            {self.narrative_summary}
            {anchor_hint}

            Generate the next scene JSON.
            """
        )

    def continuity_guardian(
        self, prev: dict[str, Any], new: dict[str, Any], scene: str
    ) -> list[str]:
        issues: list[str] = []
        prev_inv = set(prev.get("inventory") or [])
        new_inv = set(new.get("inventory") or [])
        removed = prev_inv - new_inv
        if removed and prev.get("turn", 0) > 0:
            for item in removed:
                if item.lower() not in scene.lower():
                    issues.append(
                        f"Item removed without mention: '{item}' (was in inventory)"
                    )
        if prev.get("location") and new.get("location") == prev.get("location"):
            if new.get("turn", 0) > 1 and "travel" in (new.get("last_choice") or "").lower():
                issues.append("Choice implied travel but location unchanged")
        prev_anchors = {a.get("id") for a in prev.get("anchors") or []}
        new_anchors = {a.get("id") for a in new.get("anchors") or []}
        lost = prev_anchors - new_anchors
        if lost:
            issues.append(f"Anchors dropped from state: {lost}")
        return issues

    def _detect_anchor_callbacks(self, scene: str) -> list[str]:
        callbacks: list[str] = []
        for anchor in self.world_state.get("anchors") or []:
            desc = anchor.get("description", "")
            tokens = [t for t in re.split(r"\W+", desc.lower()) if len(t) > 4]
            hits = sum(1 for t in tokens[:5] if t in scene.lower())
            if hits >= 2 or anchor.get("id", "").replace("_", " ") in scene.lower():
                callbacks.append(anchor.get("id", "unknown"))
        return callbacks

    def step(self, player_choice: str | None = None) -> TurnRecord:
        prev_state = json.loads(json.dumps(self.world_state))
        prompt = self._build_user_prompt(player_choice)
        data = self._call_llm(prompt)

        scene = data.get("scene", "")
        choices = data.get("choices", [])[:3]
        while len(choices) < 3:
            choices.append(f"Option {len(choices) + 1}")

        new_state = data.get("world_state", prev_state)
        new_state["turn"] = prev_state.get("turn", 0) + 1
        for a in data.get("new_anchors") or []:
            existing = new_state.get("anchors") or []
            if not any(x.get("id") == a.get("id") for x in existing):
                existing.append(a)
            new_state["anchors"] = existing

        issues = self.continuity_guardian(prev_state, new_state, scene)
        self.world_state = new_state
        self.narrative_summary = data.get("narrative_summary", self.narrative_summary)

        callbacks = self._detect_anchor_callbacks(scene)
        record = TurnRecord(
            turn=new_state["turn"],
            scene=scene,
            choices=choices,
            choice_taken=player_choice,
            world_state=new_state,
            narrative_summary=self.narrative_summary,
            continuity_notes=issues + list(data.get("continuity_notes") or []),
            anchor_callbacks=callbacks,
        )
        self.log.records.append(record)
        return record

    def run_autoplay(
        self,
        choices: list[str],
        on_turn: Callable[[TurnRecord], None] | None = None,
    ) -> PlaythroughLog:
        for i, choice in enumerate(choices):
            rec = self.step(None if i == 0 and not self.log.records else choice)
            if on_turn:
                on_turn(rec)
        return self.log

    def continuity_score(self) -> dict[str, Any]:
        """Score whether early anchors appear in late scenes (rubric: turn 3 → turn 9)."""
        if not self.log.records:
            return {"error": "no playthrough"}
        early_anchors: dict[str, int] = {}
        for r in self.log.records:
            for a in r.world_state.get("anchors") or []:
                t = a.get("introduced_turn", r.turn)
                if t <= 4:
                    early_anchors[a.get("id", "?")] = t
        late_records = [r for r in self.log.records if r.turn >= 8]
        hits: list[dict[str, Any]] = []
        for aid, intro in early_anchors.items():
            for r in late_records:
                if aid in r.anchor_callbacks or aid.replace("_", " ") in r.scene.lower():
                    hits.append(
                        {"anchor_id": aid, "introduced_turn": intro, "callback_turn": r.turn}
                    )
                    break
        return {
            "early_anchors": early_anchors,
            "late_turn_callbacks": hits,
            "continuity_pass": len(hits) >= 1 and len(self.log.records) >= 10,
            "guardian_issue_count": sum(len(r.continuity_notes) for r in self.log.records),
        }



## 2. Start a story


In [ ]:
GENRE = "fantasy"  # fantasy | noir | sci_fi
TITLE = "The Anchor Below"

engine = StoryEngine(title=TITLE, genre=GENRE)
print("Story:", TITLE, "| Genre:", GENRE, "| Backend:", engine.provider)


## 3. Interactive play


In [ ]:
def play_one_turn(engine: StoryEngine, choice: str | None = None):
    rec = engine.step(choice)
    display(Markdown("### Turn %d\n\n%s" % (rec.turn, rec.scene)))
    lines = "\n".join("%d. %s" % (i, c) for i, c in enumerate(rec.choices, 1))
    display(Markdown("**Choices:**\n" + lines))
    if rec.anchor_callbacks:
        display(Markdown("**Anchor callbacks:** " + ", ".join(rec.anchor_callbacks)))
    if rec.continuity_notes:
        display(Markdown("**Continuity guardian:** " + "; ".join(rec.continuity_notes)))
    return rec

play_one_turn(engine)


In [ ]:
# Edit choice text each run
play_one_turn(engine, choice="Follow the warming map west")


## 4. Autoplay — 12-turn submission log


In [ ]:
SCRIPTED_CHOICES = [
    "Step through the cracked archway",
    "Follow the warming map west",
    "Accept the ghost librarian's bargain",
    "Light the brass lantern",
    "Descend the spiral stairs",
    "Read the copper-scented map aloud",
    "Hide when footsteps approach",
    "Confront the figure at the sealed stack",
    "Return the map as promised",
    "Open the final door",
    "Claim what waits inside",
    "Walk back into the rain",
]

demo = StoryEngine(title="Autoplay — The Anchor Below", genre="fantasy")
for i, ch in enumerate(SCRIPTED_CHOICES):
    rec = demo.step(None if i == 0 else ch)
    print("--- Turn %d ---" % rec.turn)
    print(rec.scene[:500])
    print("Choices:", rec.choices)
    print("Anchors:", [a.get("id") for a in rec.world_state.get("anchors", [])])
    print()

print(json.dumps(demo.continuity_score(), indent=2))


## 5. Export playthrough & continuity check


In [ ]:
OUT = Path("playthrough_log.md")
OUT.write_text(demo.log.to_markdown(), encoding="utf-8")
print("Saved", OUT.resolve())
demo.continuity_score()


## 6. Failure analysis (complete after a real LLM run)

Document cases where the model lost state or broke continuity:

| Turn | Symptom | Likely cause | Mitigation | Result |
|------|---------|--------------|------------|--------|
| | | | | |

*Use OpenAI/Ollama for honest failures; mock mode is for structure testing only.*
